# SMPL extraction for MoonBoard clips (spike1) — 4D-Humans on Colab

Runs [4D-Humans](https://github.com/shubham-goel/4D-Humans) (HMR 2.0 + PHALP tracking)
on the `spike1` clips to get per-frame SMPL body parameters. Output: one `.pkl` per
clip with pose/shape/camera per tracked person per frame — the input for the
SMPL → 29-DOF retargeting step (NEXT_STEPS §V4).

**Runtime: GPU required** — `Runtime → Change runtime type → T4 GPU` (free tier is fine;
expect ~2–5 min per clip).

The 2D feasibility gate already passed on these clips (YOLO11-pose: ≥99% person
detection at every grade, see the spike1 README). This notebook answers the 3D question:
are the SMPL fits clean enough to retarget? Inspect for: flipped limbs, wrong
depth ordering against the wall, jitter on the reach arm.

In [ ]:
# 1) GPU sanity
!nvidia-smi -L
import torch; print('torch sees cuda:', torch.cuda.is_available())

In [ ]:
# 2) Install 4D-Humans (per its README; takes a few minutes)
!git clone https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -e .[all] -q
# PHALP (the tracker used by track.py)
!pip install git+https://github.com/brjathu/PHALP.git -q

**SMPL body model**: 4D-Humans downloads its HMR2 checkpoints automatically, but the
SMPL neutral body model requires a (free) registration at https://smpl.is.tue.mpg.de.
Download `basicModel_neutral_lbs_10_207_0_v1.0.0.pkl` and upload it when prompted by
the first run (or place it at the path the error message names).

In [ ]:
# 3) Upload the spike1 clips (or mount Drive and copy them)
from google.colab import files
import pathlib
pathlib.Path('clips').mkdir(exist_ok=True)
print('Select one or more clipXX_*.mp4 files…')
up = files.upload()
for name in up:
    pathlib.Path(name).rename(f'clips/{name}')
!ls -la clips/

In [ ]:
# 4) Track + reconstruct each clip → outputs/ .pkl with SMPL params per frame
import glob, subprocess
for clip in sorted(glob.glob('clips/*.mp4')):
    print('===', clip)
    subprocess.run(['python', 'track.py', f'video.source={clip}'], check=False)
!find outputs -name '*.pkl' | sort

In [ ]:
# 5b) Convert 4D-Humans .pkl output → retarget-ready .npz (one per clip per person)
#
# 4D-Humans PHALP output structure:
#   d = joblib.load(pkl)           -> {frame_idx: {tracked_ids, smpl, camera, ...}}
#   d[frame]['smpl'][person_idx]   -> {body_pose (69,), global_orient (3,),
#                                       betas (10,), transl (3,)}
#
# We pick the highest-confidence person per frame (usually just 1 on a MoonBoard),
# concatenate global_orient+body_pose → (T,72) and save with translation → (T,3).

import joblib, numpy as np, pathlib, glob

out_dir = pathlib.Path('spike1_npz')
out_dir.mkdir(exist_ok=True)

def _arr(x):
    return np.asarray(x, dtype=np.float32).flatten()

for pkl_path in sorted(glob.glob('outputs/**/*.pkl', recursive=True)):
    clip_name = pathlib.Path(pkl_path).stem  # e.g. "clip01_v3a"
    print(f"Converting {pkl_path} ...")
    try:
        d = joblib.load(pkl_path)
    except Exception as e:
        print(f"  SKIP (load error): {e}"); continue

    frames = sorted(d.keys())
    poses, transl = [], []
    for fid in frames:
        frame = d[fid]
        smpl_list = frame.get('smpl') or frame.get('smpl_params') or []
        if not smpl_list:
            # pad with zeros so frame count is consistent
            poses.append(np.zeros(72, dtype=np.float32))
            transl.append(np.zeros(3, dtype=np.float32))
            continue
        # pick person 0 (highest tracking confidence — PHALP sorts by conf)
        sp = smpl_list[0]
        go = _arr(sp.get('global_orient', np.zeros(3)))[:3]
        bp = _arr(sp.get('body_pose',     np.zeros(69)))[:69]
        tr = _arr(sp.get('transl',        np.zeros(3)))[:3]
        poses.append(np.concatenate([go, bp]))
        transl.append(tr)

    poses  = np.stack(poses)   # (T, 72)
    transl = np.stack(transl)  # (T, 3)
    out = out_dir / f"{clip_name}.npz"
    np.savez(out, smpl_poses=poses, smpl_trans=transl, fps=np.float32(25.0))
    print(f"  → {out}  ({len(frames)} frames)")

print("\nAll done. Files in spike1_npz/:")
for p in sorted(out_dir.glob("*.npz")):
    d2 = np.load(p)
    print(f"  {p.name}: poses {d2['smpl_poses'].shape}  trans {d2['smpl_trans'].shape}")


In [ ]:
# 5) Quick QA — load one result and sanity-check the SMPL stream
import joblib, glob, numpy as np
pkls = sorted(glob.glob('outputs/**/*.pkl', recursive=True))
d = joblib.load(pkls[0])
frames = sorted(d.keys())
print(f'{pkls[0]}: {len(frames)} frames')
f0 = d[frames[len(frames)//2]]
print('tracked people in mid frame:', len(f0.get("tracked_ids", [])))
# smpl params: global_orient (3), body_pose (69), betas (10) per person
if f0.get('smpl'):
    sp = f0['smpl'][0]
    print('body_pose shape:', np.asarray(sp['body_pose']).shape,
          ' betas:', np.round(np.asarray(sp['betas']).flatten()[:5], 2))

In [ ]:
# 6) Bundle BOTH the raw pkl outputs AND the retarget-ready npz files
!zip -rq spike1_smpl.zip outputs spike1_npz
from google.colab import files
files.download('spike1_smpl.zip')


## What to look for in the rendered videos (in `outputs/`)

- **Limb assignment** — left/right swaps on crossed arms are the classic failure.
- **Depth vs the wall** — the body should stay on the climber's side of the wall plane;
  watch the reach arm at full extension.
- **The V3 vs V6/V7 comparison** — same climber/camera/lighting, so any quality drop
  isolates movement style.
- **Feet near the pads** — the first ~1 s of each climb has occluded feet; that's
  expected noise, clip it during retargeting.

If fits look clean → proceed to V4 (SMPL → 29-DOF retargeting in MuJoCo).
If depth is noisy → the known-hold-grid correction becomes load-bearing (V3 contact
labelling first, then constrain the IK with contact points).